# ORSY-Propensity-Modell – Decision Tree mit F1-Score

Ziel:
- Vorhersage, ob ein Kunde ORSY-Kunde ist (`flag_new_orsyshelf` → 0/1).
- Binäre Klassifikation mit stark unbalancierten Klassen (~9 % ORSY).

Wir nutzen:
- **DecisionTreeClassifier** als Modell,
- **Undersampling** der Mehrheit (Nicht-ORSY),
- **F1-Score** als Hauptmetrik für die ORSY-Klasse.

**F1-Score (intuitiv):**
- kombiniert **Precision** (wie sauber sind unsere ORSY-Treffer?)  
  und **Recall** (wie viele der echten ORSY-Kunden finden wir?).
- F1 ist besonders sinnvoll, wenn eine Klasse selten ist (hier ORSY ≈ 9 %).

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils import resample

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

print("Imports erfolgreich geladen.")

Imports erfolgreich geladen.


In [2]:
# Pfad an deine Struktur anpassen (Notebook in notebooks/, Daten in data/)
data_path = "../data/dataset_wuerth.csv"

df = pd.read_csv(data_path)

# Zielvariable: ORSY (0/1)
df["orsy"] = (df["flag_new_orsyshelf"] > 0).astype(int)

print("Form des Datensatzes:", df.shape)
print("ORSY-Anteil gesamt: {:.2f}%".format(100 * df["orsy"].mean()))

df[["cust_id", "market_seg", "emp_count", "orders_count", "sales", "orsy"]].head()

Form des Datensatzes: (29493, 43)
ORSY-Anteil gesamt: 9.35%


,cust_id,market_seg,emp_count,orders_count,sales,orsy
0,40071,9,1,10,2445.11,0
1,66107,2,10,67,14070.62,1
2,33010,21,8,25,5893.92,0
3,96845,7,4,14,1823.72,0
4,64230,26,8,122,23654.08,1


## Filter & Feature-Set

**Filter:**
- `emp_count > 0` → Kunden mit 0 Mitarbeitern werden ausgeschlossen.

**Features im Modell:**
- `market_seg` (Marktsegment)
- `emp_count` (Mitarbeiterzahl, > 0)
- `orders_count` (Anzahl Bestellungen)
- `sales` (Gesamtumsatz)
- `rev_salesrep`
- `rev_branch_office`
- `rev_ebusiness`
- `rev_internal_staff`

Nicht berücksichtigt:
- `rev_others`, Region/District, digitale Flags, weitere Detailvariablen.

In [3]:
# Filter: nur Kunden mit emp_count > 0
df_model = df[df["emp_count"] > 0].copy()
print("Nach Filter emp_count > 0:", df_model.shape)

# Feature-Liste nach deiner Vorgabe
features = [
    "market_seg",
    "emp_count",
    "orders_count",
    "sales",
    "rev_salesrep",
    "rev_branch_office",
    "rev_ebusiness",
    "rev_internal_staff",
]

X = df_model[features].fillna(0)
y = df_model["orsy"]

print("Features:", features)
print("X-Form:", X.shape, " y-Form:", y.shape)
print("ORSY-Anteil nach Filter: {:.2f}%".format(100 * y.mean()))

Nach Filter emp_count > 0: (29135, 43)
Features: ['market_seg', 'emp_count', 'orders_count', 'sales', 'rev_salesrep', 'rev_branch_office', 'rev_ebusiness', 'rev_internal_staff']
X-Form: (29135, 8)  y-Form: (29135,)
ORSY-Anteil nach Filter: 9.44%


## Undersampling der Nicht-ORSY-Kunden

Da der ORSY-Anteil nur bei ca. 9 % liegt, balancieren wir die Klassen:

- Mehrheit: Nicht-ORSY (0)
- Minderheit: ORSY (1)

Wir machen:
- **Random Undersampling** der Mehrheit auf etwa das **3-fache** der Minderheit,
- dadurch bleiben die Daten realistisch, aber das Modell „sieht“ genug ORSY-Beispiele.

In [4]:
df_major = df_model[df_model["orsy"] == 0]
df_minor = df_model[df_model["orsy"] == 1]

print("Originale Verteilung (nach emp_count > 0):")
print(df_model["orsy"].value_counts(normalize=True).rename("Anteil"))

# Zielgröße für Mehrheit (z.B. 3:1 Verhältnis)
target_major_n = min(len(df_major), len(df_minor) * 3)

df_major_down = resample(
    df_major,
    replace=False,
    n_samples=target_major_n,
    random_state=42
)

df_bal = pd.concat([df_major_down, df_minor]).sample(frac=1, random_state=42)

print("\nNeue Verteilung nach Undersampling:")
print(df_bal["orsy"].value_counts(normalize=True).rename("Anteil"))

X_bal = df_bal[features].fillna(0)
y_bal = df_bal["orsy"]
print("Balancierter Datensatz:", X_bal.shape)

Originale Verteilung (nach emp_count > 0):
orsy
0    0.905577
1    0.094423
Name: Anteil, dtype: float64

Neue Verteilung nach Undersampling:
orsy
0    0.75
1    0.25
Name: Anteil, dtype: float64
Balancierter Datensatz: (11004, 8)
